# Data split and preprocessing

The data is split chronologically into train (70%), validation (15%), and test (15%) partitions. The preprocessing pipeline is then applied independently to each partition to keep the split boundaries clear and avoid sharing fitted information between partitions.

The pipeline uses `arrival_planned_time` and falls back to `departure_planned_time` to create one `planned_time` column. This keeps rows with a usable schedule timestamp while providing a consistent value for chronological ordering. The original arrival and departure schedule columns are removed after this value is created.

Delay values below -1 minute or above 180 minutes are treated as extreme outliers and removed. These thresholds preserve normal early and late arrivals while excluding values that are unlikely to represent reliable observations.

A `first_station` feature is derived from rows where `train_line_station_num == 1`. IC and ICE services are matched by `train_number`, while RE services are matched by `line_number`. Rows without a derivable first station are removed because this feature is required for the analysis.

Rows without a line number are removed for RE services because the line number is needed to identify their route. Missing line numbers for IC and ICE are retained unless the optional replacement is enabled. Missing final destinations can optionally be replaced with the current station because a missing final destination indicates that the current stop is final.

The `time` date field is also converted into a `day_of_week` feature for later modelling.

In [12]:
import sys
from pathlib import Path

import pandas as pd
import importlib

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import data_utils
importlib.reload(data_utils)
from data_utils import add_delay_target, chronological_split, load_monthly_data
import preprocessing
importlib.reload(preprocessing)
from preprocessing import add_planned_time_features, preprocess_splits

columns = [
    "delay_in_min",
    "arrival_is_canceled",
    "arrival_planned_time",
    "departure_planned_time",
    "time",
    "train_type",
    "line_number",
    "station_name",
    "train_number",
    "train_line_station_num",
    "train_line_ride_id",
    "final_destination_station",
]
labelled = add_delay_target(
    load_monthly_data(
        PROJECT_ROOT / "data",
        columns,
        require_planned_time=False,
    )
)
labelled = add_planned_time_features(labelled, add_weekday=False)
labelled = labelled.dropna(subset=["planned_time"]).copy()
train, validation, test = chronological_split(labelled, sort_column="planned_time")

pd.DataFrame(
    {
        "partition": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
    }
).assign(percent=lambda frame: frame["rows"] / len(labelled) * 100)

preprocessed = preprocess_splits(
    {"train": train, "validation": validation, "test": test},
    fill_missing_line_number=None,
    fill_missing_final_destination=False,
)

pd.DataFrame(
    {
        "partition": list(preprocessed),
        "rows_before": [len(train), len(validation), len(test)],
        "rows_after": [len(preprocessed[name]) for name in preprocessed],
    }
).assign(
    removed=lambda frame: frame["rows_before"] - frame["rows_after"]
)

first_station_coverage = pd.DataFrame(
    {
        "partition": list(preprocessed),
        "rows": [len(preprocessed[name]) for name in preprocessed],
        "missing_first_station_count": [
            preprocessed[name]["first_station"].isna().sum()
            for name in preprocessed
        ],
    }
)
first_station_coverage["missing_percent"] = (
    first_station_coverage["missing_first_station_count"]
    / first_station_coverage["rows"]
    * 100
).round(2)
display(first_station_coverage)

for name, partition in preprocessed.items():
    print(f"\\n{name} head:")
    display(partition.head())

,partition,rows,missing_first_station_count,missing_percent
0,train,10122747,0,0.0
1,validation,2160671,0,0.0
2,test,2160340,0,0.0


\ntrain head:


,delay_in_min,arrival_is_canceled,time,train_type,line_number,station_name,train_number,train_line_station_num,train_line_ride_id,final_destination_station,source_file,target,planned_time,day_of_week,first_station
0,71.0,False,2026-01-01 00:31:00,RE,RE30,Frankfurt (Main) Hbf,4168,1,5406595945530653202,Kassel Hbf,data-2026-01.parquet,severe_or_cancelled,2025-12-31 23:20:00,Thursday,Frankfurt (Main) Hbf
1,47.0,False,2026-01-01 00:08:00,RE,RE70,Kiel Hbf,11236,8,4429131433570418178,NaN,data-2026-01.parquet,severe_or_cancelled,2025-12-31 23:21:00,Thursday,Mannheim Hbf
2,26.0,False,2026-01-01 00:03:00,ICE,NaN,Mannheim Hbf,677,12,7477797110160472438,Karlsruhe Hbf,data-2026-01.parquet,delayed,2025-12-31 23:29:00,Thursday,Berlin Ostbahnhof
3,33.0,False,2026-01-01 00:05:00,ICE,NaN,Frankfurt am Main Flughafen Fernbahnhof,319,9,-8088851598754426904,Frankfurt(Main)Hbf,data-2026-01.parquet,severe_or_cancelled,2025-12-31 23:29:00,Thursday,Köln Hbf
4,37.0,False,2026-01-01 00:06:00,RE,RE200,Ulm Hbf,70498,1,7132089998541061809,Wendlingen(Neckar),data-2026-01.parquet,severe_or_cancelled,2025-12-31 23:29:00,Thursday,Ulm Hbf


\nvalidation head:


,delay_in_min,arrival_is_canceled,time,train_type,line_number,station_name,train_number,train_line_station_num,train_line_ride_id,final_destination_station,source_file,target,planned_time,day_of_week,first_station
10231017,174.0,False,2026-07-10 23:36:00,RE,RE42,Mönchengladbach Hbf,10158,21,-2686774313141415833,NaN,data-2026-07.parquet,severe_or_cancelled,2026-07-10 20:42:00,Friday,Essen Hbf
10231018,0.0,False,2026-07-10 20:43:00,RE,RE32,Pegnitz,3092,4,-2570326620499788708,NaN,data-2026-07.parquet,on_time,2026-07-10 20:43:00,Friday,Pegnitz
10231019,0.0,False,2026-07-10 20:43:00,RE,RE3,Wilmersdorf (b Angermünde),3316,24,-6400209055042475724,Anklam,data-2026-07.parquet,on_time,2026-07-10 20:43:00,Friday,Erzingen (Baden)
10231020,0.0,False,2026-07-10 20:43:00,RE,RE14,Mainz Hbf,4498,4,2252869086695458865,NaN,data-2026-07.parquet,on_time,2026-07-10 20:43:00,Friday,Saalfeld (Saale)
10231021,0.0,False,2026-07-10 20:43:00,RE,RE41,Haltern am See,16243,5,8172722712721445420,NaN,data-2026-07.parquet,on_time,2026-07-10 20:43:00,Friday,Neustadt (Waldnaab)


\ntest head:


,delay_in_min,arrival_is_canceled,time,train_type,line_number,station_name,train_number,train_line_station_num,train_line_ride_id,final_destination_station,source_file,target,planned_time,day_of_week,first_station
12423378,12.0,False,2026-08-21 17:22:00,ICE,NaN,Eisenach Hbf,1653,5,-2999068967832918015,Dresden Hbf,data-2026-08.parquet,delayed,2026-08-21 17:08:00,Friday,Frankfurt am Main Flughafen Fernbahnhof
12423379,16.0,False,2026-08-21 17:26:00,ICE,NaN,Erfurt Hbf,2823,2,-1526327176180092704,Berlin Gesundbrunnen,data-2026-08.parquet,delayed,2026-08-21 17:08:00,Friday,Nürnberg Hbf
12423380,16.0,False,2026-08-21 17:26:00,ICE,NaN,Erfurt Hbf,751,2,-4557894753275967683,Berlin Gesundbrunnen,data-2026-08.parquet,delayed,2026-08-21 17:08:00,Friday,Frankfurt (Main) Hbf
12423381,1.0,False,2026-08-21 17:27:00,RE,RE10b,Grombach,19280,8,4394384259837185267,Mannheim Hbf,data-2026-08.parquet,on_time,2026-08-21 17:08:00,Friday,Mannheim Hbf
12423382,27.0,False,2026-08-21 17:35:00,RE,RE20,Magdeburg-Eichenweiler,16943,18,6334934090237196314,Schönebeck-Bad Salzelmen,data-2026-08.parquet,delayed,2026-08-21 17:08:00,Friday,Schönebeck-Bad Salzelmen
